# Chapter 10 · Data is the cheapest efficiency lever — Lab 10

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omar-florez/training-efficient-llms/blob/main/labs/lab10_data_pipeline.ipynb)

**The minimal data pipeline, end to end, on a CPU.** You will stream a slice of FineWeb-Edu from Hugging Face, pin its revision, deduplicate it exactly and approximately (MinHash-LSH with FineWeb's 14×8 parameters), run the heuristic filters of Table 10.1 and measure what each removes, look at the educational-classifier score distribution that produced the 1.3T-token set, decontaminate against GSM8K and MMLU with 13-gram tripwires, score difficulty with a small reference model (§10.6), and write the dataset card with a lineage record. Every number you print here is one you can compare with the chapter's.

Runs on the free Colab CPU in ~10 minutes with `N_DOCS = 20_000`. Section 7 (difficulty scoring) is faster with a GPU runtime but works on CPU with a smaller sample.

*Omar U. Florez, PhD · Efficient Training of LLMs · Lab 10*

In [ ]:
%pip -q install datasets datasketch transformers torch pyarrow
import re, json, hashlib, time, collections, itertools
import numpy as np, matplotlib.pyplot as plt
from datasets import load_dataset

NAVY, BLUE, LIGHT, AMBER, GRAY = "#17406b", "#3f74b8", "#9dbfe4", "#b45309", "#5c5c5c"
plt.rcParams.update({"figure.facecolor":"white","axes.facecolor":"white","axes.edgecolor":GRAY,
    "axes.labelcolor":"#1a1a1a","axes.grid":True,"grid.color":"#e3eaf3","grid.linewidth":0.8,
    "axes.spines.top":False,"axes.spines.right":False,"font.size":11,"figure.dpi":110})

N_DOCS = 20_000          # raise on a bigger machine; the chapter's numbers were measured on 15T tokens
LINEAGE = {}             # the record that becomes the dataset card (§10.2)

## 1 · Stream from the warehouse, and pin the revision

Chapter 10.2: a dataset is a *revision*, not a name. We stream (no 10BT download), take the first `N_DOCS` documents, and record the dataset id, config, split, and git revision so the slice is reproducible. FineWeb-Edu rows carry the columns the rest of the lab uses: `text`, `url`, `dump`, `language_score`, `score` (the educational classifier, 0–5), and `token_count`.

In [ ]:
DATASET, CONFIG, SPLIT = "HuggingFaceFW/fineweb-edu", "sample-10BT", "train"
stream = load_dataset(DATASET, name=CONFIG, split=SPLIT, streaming=True)
t0 = time.time()
docs = list(itertools.islice(stream, N_DOCS))
print(f"{len(docs):,} docs in {time.time()-t0:.0f}s")

# pin the exact revision: streaming datasets expose it through the builder's info; fall back to the Hub API
try:
    from huggingface_hub import HfApi
    rev = HfApi().dataset_info(DATASET).sha
except Exception as e:
    rev = "unknown (offline?)"
LINEAGE["source"] = {"dataset": DATASET, "config": CONFIG, "split": SPLIT, "revision": rev,
                     "n_docs_taken": len(docs), "columns": list(docs[0].keys())}
print(json.dumps(LINEAGE["source"], indent=1))

d = docs[0]
print("\n--- one document ---")
print("url:", d["url"]); print("dump:", d["dump"], "| edu score:", d["score"], "| lang score:", round(d["language_score"],3), "| tokens:", d["token_count"])
print(d["text"][:600], "…")

In [ ]:
toks = np.array([d["token_count"] for d in docs])
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].hist(np.log10(toks), bins=50, color=BLUE); ax[0].set_xlabel("log10 tokens per document"); ax[0].set_ylabel("docs")
ax[0].axvline(np.log10(np.median(toks)), color=AMBER, lw=2, label=f"median {np.median(toks):.0f}"); ax[0].legend()
dumps = collections.Counter(d["dump"] for d in docs)
ax[1].barh([k for k,_ in dumps.most_common(8)][::-1], [v for _,v in dumps.most_common(8)][::-1], color=LIGHT)
ax[1].set_xlabel("docs"); ax[1].set_title("Common Crawl dumps in this slice (per-dump dedup lives here)")
plt.tight_layout(); print(f"total tokens in slice: {toks.sum():,}")

## 2 · Deduplication: exact, then near (MinHash-LSH, 14 × 8)

Chapter 10.3. Exact dedup is a normalized SHA-1 and costs nothing. Near-dedup is MinHash on word 5-grams with 112 hashes arranged as b = 14 bands × r = 8 rows, which gives the S-curve $P(\text{candidate}) = 1 - (1 - J^{r})^{b}$: a pair at J = 0.75 is flagged 77% of the time, at 0.85 98.8% — the numbers in Figure 10.2. We first reproduce the curve, then run the index.

In [ ]:
b, r = 14, 8
J = np.linspace(0, 1, 201); P = 1 - (1 - J**r)**b
for j in (0.70, 0.75, 0.80, 0.85):
    print(f"J={j:.2f} → P(flagged) = {1-(1-j**r)**b:.3f}")
plt.figure(figsize=(6,3)); plt.plot(J, P, color=NAVY, lw=2, label="FineWeb: 112 hashes, b=14, r=8")
Pr = 1 - (1 - J**20)**450; plt.plot(J, Pr, color=AMBER, lw=1.5, ls="--", label="RefinedWeb: 9,000 hashes, b=450, r=20")
plt.xlabel("Jaccard similarity J"); plt.ylabel("P(candidate)"); plt.legend(); plt.title("Figure 10.2, recomputed");

In [ ]:
from datasketch import MinHash, MinHashLSH
WS = re.compile(r"\s+")
def normalize(t): return WS.sub(" ", t.lower()).strip()
def shingles(text, n=5):
    w = re.findall(r"\w+", text.lower()); return {" ".join(w[i:i+n]) for i in range(max(1, len(w)-n+1))}

# exact
seen, exact_dups = set(), 0
for d in docs:
    h = hashlib.sha1(normalize(d["text"]).encode()).hexdigest()
    d["exact_dup"] = h in seen; exact_dups += d["exact_dup"]; seen.add(h)
print(f"exact duplicates: {exact_dups} / {len(docs)} = {exact_dups/len(docs):.2%}")

# near (per-source, per-snapshot is the chapter's lesson; our slice is one source so we index it all)
t0 = time.time()
lsh = MinHashLSH(threshold=0.75, num_perm=b*r, params=(b, r))
near_dups, pairs = 0, []
for i, d in enumerate(docs):
    if d["exact_dup"]: d["near_dup"] = False; continue
    m = MinHash(num_perm=b*r)
    for s in shingles(d["text"]): m.update(s.encode())
    hits = lsh.query(m)
    d["near_dup"] = bool(hits)
    if hits: near_dups += 1; pairs.append((hits[0], i))
    else: lsh.insert(i, m)
print(f"near duplicates: {near_dups} = {near_dups/len(docs):.2%}  ({time.time()-t0:.0f}s)")
LINEAGE["dedup"] = {"exact": "sha1(normalized)", "near": f"minhash 5-gram k={b*r} b={b} r={r} thr=0.75",
                    "scope": "this slice (per source, per snapshot in production)",
                    "removed_exact": exact_dups, "removed_near": near_dups}
if pairs:
    a, c = pairs[0]
    print("\n--- a flagged pair ---\n[A]", docs[a]["text"][:300].replace("\n"," "), "\n[B]", docs[c]["text"][:300].replace("\n"," "))

> **Exercise 2.1.** Compute the true Jaccard similarity of the flagged pairs (`len(A & B) / len(A | B)` on the shingle sets) and histogram it. How many fall below 0.75? Those are the S-curve's left tail — the false positives you are paying for with the cheap curve.

## 3 · Heuristic filters, with what each removes

Chapter 10.4, Table 10.1: the Gopher/C4/FineWeb rules, applied in order, each one measured. Note that FineWeb-Edu's rows have already been through FineWeb's pipeline, so most rules remove little here — the point is the *measurement harness*. We keep the C4 terminal-punctuation rule in the harness but **do not apply it**, as the chapter explains (it would cut ~30% of good text).

In [ ]:
def gopher_quality(t):
    w = re.findall(r"\S+", t); n = len(w)
    if n < 50 or n > 100_000: return "word_count"
    mwl = np.mean([len(x) for x in w])
    if mwl < 3 or mwl > 10: return "mean_word_len"
    if sum(x.startswith(("•","-","*")) for x in t.splitlines()) / max(1, t.count("\n")+1) > 0.9: return "bullets"
    if sum(l.rstrip().endswith("...") or l.rstrip().endswith("…") for l in t.splitlines()) / max(1, t.count("\n")+1) > 0.3: return "ellipsis"
    if sum(x.isalpha() for x in t) / max(1, len(t)) < 0.6: return "alpha_ratio"    # ≈ the symbol / word ratio rule
    return None
def gopher_repetition(t):
    lines = [l.strip() for l in t.splitlines() if l.strip()]
    if lines and len(set(lines)) / len(lines) < 0.7: return "dup_lines"
    w = re.findall(r"\w+", t.lower())
    if len(w) > 50:
        tri = collections.Counter(zip(w, w[1:], w[2:])); top = tri.most_common(1)[0][1]
        if top * 3 / len(w) > 0.20: return "top_trigram"
    return None
def fineweb_custom(t):
    lines = [l for l in t.splitlines() if l.strip()]
    if lines and sum(len(l) < 30 for l in lines) / len(lines) > 0.67: return "short_lines"   # the ~22% custom rule family
    return None
def c4_terminal(t):   # measured, NOT applied
    lines = [l for l in t.splitlines() if l.strip()]
    return lines and sum(l.rstrip()[-1] in ".!?\"'" for l in lines) / len(lines) < 0.5

removed = collections.Counter(); survivors = []
for d in docs:
    if d["exact_dup"] or d["near_dup"]: continue
    if d["language_score"] < 0.65: removed["language<0.65"] += 1; continue
    for rule in (gopher_quality, gopher_repetition, fineweb_custom):
        why = rule(d["text"])
        if why: removed[why] += 1; break
    else:
        survivors.append(d)
would_cut = sum(bool(c4_terminal(d["text"])) for d in survivors)
print("after dedup:", len(docs) - exact_dups - near_dups, "| after filters:", len(survivors))
for k, v in removed.most_common(): print(f"  {k:>16}: {v:6d}  ({v/len(docs):.2%})")
print(f"C4 terminal-punctuation would ALSO cut {would_cut/len(survivors):.1%} of survivors — kept out on purpose (Table 10.1)")
LINEAGE["filters"] = {"applied": ["language_score>=0.65", "gopher_quality", "gopher_repetition", "short_lines"],
                      "removed": dict(removed), "measured_not_applied": {"c4_terminal_punct": would_cut}}

## 4 · The classifier tier: what a threshold costs in tokens

Chapter 10.4: FineWeb-Edu's educational score (Llama-3-70B labels → distilled regressor) at threshold 3 keeps 1.3T of 15T tokens; threshold 2 keeps 5.4T. The `score` column is that classifier. Here you see the curve the chapter describes — tokens retained vs threshold — on your slice. (sample-10BT is already filtered at ≥ 3, so the left of the curve is flat; the exercise below gets the unfiltered version.)

In [ ]:
sc = np.array([d["score"] for d in survivors]); tk = np.array([d["token_count"] for d in survivors])
thr = np.linspace(0, 5, 51); kept = [tk[sc >= t].sum() / tk.sum() for t in thr]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].hist(sc, bins=40, color=LIGHT, edgecolor=NAVY); ax[0].set_xlabel("educational score (0–5)"); ax[0].set_ylabel("docs")
ax[1].plot(thr, kept, color=NAVY, lw=2); ax[1].set_xlabel("threshold"); ax[1].set_ylabel("fraction of tokens kept")
for t, lab in ((2, "thr 2 → 5.4T on FineWeb"), (3, "thr 3 → 1.3T (FineWeb-Edu)")):
    ax[1].axvline(t, color=AMBER, ls="--"); ax[1].text(t+0.05, 0.5, lab, rotation=90, color=AMBER, va="center")
plt.tight_layout()
LINEAGE["quality"] = {"classifier": "HuggingFaceFW/fineweb-edu-classifier (score column)", "threshold": 3,
                      "score_mean": float(sc.mean())}

> **Exercise 4.1.** Swap the source for `HuggingFaceFW/fineweb` (config `sample-10BT`, unfiltered by the classifier) and score 2,000 documents with `HuggingFaceFW/fineweb-edu-classifier` from the Hub. Reproduce the retained-token curve from 0 to 5 and read off the thresholds that give 10% and 35% of the tokens — FineWeb's 1.3T and 5.4T.

## 5 · Decontaminate last: 13-gram tripwires from GSM8K and MMLU

Chapter 10.4's last filter and Chapter 11's first gate. Every item of the evaluation suite becomes a set of 13-grams; a document that shares one is removed, and the *removal count per source* is logged, because a spike is a leak you want to know about.

In [ ]:
def ngrams(text, n=13):
    w = text.lower().split(); return {" ".join(w[i:i+n]) for i in range(len(w)-n+1)}
trip = set()
gsm = load_dataset("openai/gsm8k", "main", split="test")
for it in gsm: trip |= ngrams(it["question"] + " " + it["answer"])
mmlu = load_dataset("cais/mmlu", "all", split="test", streaming=True)
for it in itertools.islice(mmlu, 5000): trip |= ngrams(it["question"] + " " + " ".join(it["choices"]))
print(f"{len(trip):,} tripwire 13-grams")

clean, hits = [], collections.Counter()
for d in survivors:
    if ngrams(d["text"]) & trip: hits[d["dump"]] += 1
    else: clean.append(d)
print(f"contaminated: {len(survivors)-len(clean)} / {len(survivors)}  by dump: {dict(hits)}")
LINEAGE["decontam"] = {"method": "13-gram overlap", "suite": "gsm8k/test + mmlu/test[:5000]", "removed": len(survivors)-len(clean)}

> **Exercise 5.1.** Lower n to 8 and re-run. How many more documents are removed, and are they leaks or coincidences? Read five of them. This is the precision/recall trade the chapter mentions; 13 is the GPT-3 convention, and for short benchmark items (MMLU stems) teams often use 8–10.

## 6 · The funnel, on your slice

Figure 10.1 with your numbers. Compare the shape to the chapter's — on FineWeb-Edu input, the big removals already happened upstream.

In [ ]:
stages = [("streamed", len(docs)), ("exact dedup", len(docs)-exact_dups), ("near dedup", len(docs)-exact_dups-near_dups),
          ("heuristics", len(survivors)), ("decontaminated", len(clean))]
fig, ax = plt.subplots(figsize=(7, 3.2))
for i, (name, n) in enumerate(stages):
    ax.barh(len(stages)-i, n, color=[LIGHT, LIGHT, BLUE, NAVY, AMBER][i]); ax.text(n+50, len(stages)-i, f"{name}: {n:,}", va="center")
ax.set_yticks([]); ax.set_xlim(0, len(docs)*1.35); ax.set_xlabel("documents"); ax.set_title("Your funnel (Figure 10.1, local edition)");

## 7 · Difficulty scoring with a small reference model (§10.6)

Chapter 10.6: $\mathrm{PPL}_{\theta_{\text{ref}}}(x) = \exp(-\frac{1}{T}\sum_t \log p(x_t \mid x_{<t}))$ under a 135M reference model, used two ways — a two-sided prune of the tails, and (optionally) an easy-first warm-up ordering. On CPU score 300 documents; on a GPU raise `N_SCORE`.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
dev = "cuda" if torch.cuda.is_available() else "cpu"
N_SCORE = 2000 if dev == "cuda" else 300
tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M"); tok.pad_token = tok.eos_token
ref = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-135M", torch_dtype=torch.float32).to(dev).eval()

@torch.no_grad()
def log_ppl(texts, max_len=512):
    b = tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=max_len).to(dev)
    logits = ref(**b).logits[:, :-1]
    tgt, mask = b.input_ids[:, 1:], b.attention_mask[:, 1:]
    nll = torch.nn.functional.cross_entropy(logits.transpose(1, 2), tgt, reduction="none")
    return ((nll * mask).sum(1) / mask.sum(1)).cpu().numpy()

t0 = time.time(); scores = []
for i in range(0, N_SCORE, 8):
    scores.extend(log_ppl([d["text"] for d in clean[i:i+8]]))
for d, s in zip(clean, scores): d["difficulty"] = float(s)
print(f"scored {len(scores)} docs in {time.time()-t0:.0f}s on {dev}")
lo, hi = np.percentile(scores, [3, 97])
plt.figure(figsize=(6,3)); plt.hist(scores, bins=40, color=LIGHT, edgecolor=NAVY)
plt.axvline(lo, color=AMBER, ls="--", label="drop below (templates)"); plt.axvline(hi, color=AMBER, ls="--", label="drop above (noise)")
plt.xlabel("log PPL under SmolLM2-135M"); plt.legend(); plt.title("two-sided perplexity prune (§10.6)")
order = np.argsort(scores)
print("\nEASIEST:", clean[order[0]]["text"][:200].replace("\n"," "), "\n\nHARDEST:", clean[order[-1]]["text"][:200].replace("\n"," "))
LINEAGE["difficulty"] = {"reference": "HuggingFaceTB/SmolLM2-135M", "scored": len(scores), "prune_percentiles": [3, 97]}

> **Exercise 7.1 (distilled scorer).** Fit a `sklearn` ridge regression from TF-IDF features (or sentence embeddings) to `difficulty` on 80% of the scored docs; report Spearman ρ on the rest. That is the student of §10.6 in miniature — how close to ρ = 0.9 can a model that costs microseconds get?
>
> **Exercise 7.2 (pacing).** Sort `clean` by `difficulty`, implement $g(t)=\sqrt{t}$ and the warm-up schedule of Figure 10.5, and print the mean difficulty of the batch at t = 0.1, 0.3, 0.6.

## 8 · Write the dataset card

Chapter 10.2's rule: the lineage record *is* the card. We save the cleaned slice as Parquet and a README that a colleague (or Chapter 11's audit) can act on.

In [ ]:
import pyarrow as pa, pyarrow.parquet as pq, pathlib
out = pathlib.Path("fineweb_edu_lab10"); out.mkdir(exist_ok=True)
keep_cols = ["id", "url", "dump", "score", "language_score", "token_count", "text"]
table = pa.Table.from_pylist([{k: d.get(k) for k in keep_cols} | {"difficulty": d.get("difficulty")} for d in clean])
pq.write_table(table, out / "data.parquet")
LINEAGE["output"] = {"n_docs": len(clean), "n_tokens": int(sum(d["token_count"] for d in clean)), "file": "data.parquet"}
card = f'''# fineweb-edu-lab10 (revision of {LINEAGE["source"]["dataset"]} · {LINEAGE["source"]["config"]})
Built with Lab 10 of *Efficient Training of LLMs*. Stages, in order, with what each removed:

```json
{json.dumps(LINEAGE, indent=1)}
```

Decontaminated against GSM8K (test) and MMLU (test, first 5,000) by 13-gram overlap — **do not** report those benchmarks for a model trained on this without re-running Chapter 11's membership and completion tests.
'''
(out / "README.md").write_text(card); print(card[:1200])

## What you reproduced

* the LSH S-curve and FineWeb's 56 / 77 / 92 / 98.8% catch probabilities (Figure 10.2), then a real MinHash index with those parameters;
* the heuristic layer as a *measurement harness* — every rule with its removal fraction, and the C4 rule measured but not applied (Table 10.1);
* the classifier-threshold curve behind the 1.3T / 5.4T numbers;
* 13-gram decontamination, logged per source, as the last filter;
* a perplexity difficulty score from a 135M reference model and the two-sided prune (§10.6);
* a dataset card whose body is the lineage record.

Next: **Lab 11** audits this slice — is it relevant to the target suite (BETR-lite), is it good (judge calibration), and does the base model already know it (Min-K%)?